# 3. ColBERT Embeddings: Late Interaction Magic ✨

In this notebook, we'll implement ColBERT (Contextualized Late Interaction over BERT) - the game-changing approach that uses **token-level embeddings** instead of compressing entire documents into single vectors.

## Why ColBERT Changes Everything

Traditional dense retrieval (notebook 2) compresses a 50-word restaurant review into a single 384-dimensional vector. That's like trying to summarize a movie with one emoji! 🎬→😀

ColBERT keeps **every token's embedding** and uses **MaxSim** to find the best token-to-token matches. This preserves nuanced details that get lost in traditional RAG.

## Key Concepts
- **Late Interaction**: Embeddings interact at search time, not indexing time
- **Token-level Matching**: Every word gets its own vector
- **MaxSim Operation**: For each query token, find its best match across all document tokens
- **Multi-vector Storage**: LanceDB stores multiple embeddings per document

In [2]:
# Setup environment
import sys
sys.path.append("../..")
from setup import *

# Verify we're in the right place
print(f"📂 Working in: {os.getcwd()}")
print(f"🎯 Project root: {os.getenv('PROJECT_ROOT')}")
print(f"📊 Data directory: {os.getenv('DATA_DIR')}")

✅ Loaded environment from: /Users/luvsuneja/Documents/repos/advanced-rag-experimentation/notebooks/ColBERT/../../.env
📂 Working directory: /Users/luvsuneja/Documents/repos/advanced-rag-experimentation
🎯 Project root: /Users/luvsuneja/Documents/repos/advanced-rag-experimentation
📊 Data directory: /Users/luvsuneja/Documents/repos/advanced-rag-experimentation/data
🔧 Device: mps
📁 Environment variables loaded: 82
📂 Working in: /Users/luvsuneja/Documents/repos/advanced-rag-experimentation
🎯 Project root: /Users/luvsuneja/Documents/repos/advanced-rag-experimentation
📊 Data directory: /Users/luvsuneja/Documents/repos/advanced-rag-experimentation/data


## 1. Install and Import ColBERT Dependencies

We'll use **PyLate** - a modern Python library for ColBERT implementation that's actively maintained and optimized.

In [4]:
# Install PyLate if not already installed
try:
    import pylate
    print("✅ PyLate already installed")
except ImportError:
    print("📦 Installing PyLate...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pylate"])
    import pylate
    print("✅ PyLate installed successfully")

# Import required libraries
import torch
from pylate import models, retrieve
import lancedb
from pathlib import Path
import json
from typing import List, Dict, Tuple
import time

print(f"🔧 PyLate version: {pylate.__version__ if hasattr(pylate, '__version__') else 'unknown'}")
print(f"🔧 PyTorch version: {torch.__version__}")
print(f"🖥️  Using device: {get_device()}")

✅ PyLate already installed
🔧 PyLate version: unknown
🔧 PyTorch version: 2.2.2
🖥️  Using device: mps


## 2. Load ColBERT Model

We'll use the same base model (all-MiniLM-L6-v2) but with ColBERT's token-level approach instead of sentence-level compression.

In [11]:
dir(colbert_model)

['T_destination',
 '__add__',
 '__annotations__',
 '__call__',
 '__class__',
 '__delattr__',
 '__delitem__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattr__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__iadd__',
 '__imul__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__len__',
 '__lt__',
 '__module__',
 '__mul__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__rmul__',
 '__setattr__',
 '__setitem__',
 '__setstate__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_apply',
 '_backward_hooks',
 '_backward_pre_hooks',
 '_buffers',
 '_call_impl',
 '_compiled_call_impl',
 '_create_model_card',
 '_encode_multi_process_worker',
 '_eval_during_training',
 '_first_module',
 '_forward_hooks',
 '_forward_hooks_always_called',
 '_forward_hooks_with_kwargs',
 '_forward_pre_hooks',
 '_forward_pre_hooks_with_kwargs',
 '_get_backward_hooks',
 '_get_backward_pre_h

128


In [15]:
# Load ColBERT model
model_name = os.getenv('COLBERT_MODEL_NAME', 'sentence-transformers/all-MiniLM-L6-v2')
device = get_device()

print(f"🤖 Loading ColBERT model: {model_name}")
print(f"🖥️  Device: {device}")

start_time = time.time()

# Initialize ColBERT model with PyLate
colbert_model = models.ColBERT(
    model_name_or_path=model_name,
    device=device
)

load_time = time.time() - start_time
print(f"✅ ColBERT model loaded in {load_time:.2f} seconds")
print(f"📏 Embedding dimension: {colbert_model.get_sentence_embedding_dimension()}")
print(f"📚 Max sequence length: {colbert_model.get_max_seq_length()}")

🤖 Loading ColBERT model: sentence-transformers/all-MiniLM-L6-v2
🖥️  Device: mps


The checkpoint does not contain a linear projection layer. Adding one with output dimensions (384, 128).
Created a PyLate model from base encoder.
The tokenizer does not support resizing the token embeddings, the prefixes token have not been added to vocabulary.


✅ ColBERT model loaded in 4.37 seconds
📏 Embedding dimension: 128
📚 Max sequence length: 256


## 3. Load Restaurant Data

Load the same restaurant reviews we used in the dense embedding notebook.

In [16]:
# Load restaurant reviews
reviews_path = os.getenv('RESTAURANT_REVIEWS_CSV')
print(f"📖 Loading reviews from: {reviews_path}")

df = pd.read_csv(reviews_path)
print(f"📊 Loaded {len(df)} reviews")
print(f"📋 Columns: {list(df.columns)}")

# Display sample review to understand the data
print("\n🍝 Sample review:")
sample = df.iloc[0]
print(f"Restaurant: {sample['restaurant']}")
print(f"Review: {sample['review'][:200]}...")
print(f"Rating: {sample['rating']}/5")

# Prepare documents for embedding
documents = df['review'].tolist()
print(f"\n📄 {len(documents)} documents ready for ColBERT embedding")

📖 Loading reviews from: /Users/luvsuneja/Documents/repos/advanced-rag-experimentation/data/restaurant_reviews.csv
📊 Loaded 12 reviews
📋 Columns: ['id', 'restaurant', 'review', 'reviewer', 'rating']

🍝 Sample review:
Restaurant: Mario's Bistro
Review: OMG this little Italian place is a hidden gem! 😍 Went there last night with my boyfriend and we sat on their adorable outdoor patio with all the string lights - so romantic! The pasta was absolutely i...
Rating: 5/5

📄 12 documents ready for ColBERT embedding


## 4. Create ColBERT Embeddings

This is where the magic happens! Unlike dense embeddings that create one vector per document, ColBERT creates **multiple vectors per document** - one for each token.

### Token-Level Analysis
Let's first examine how ColBERT tokenizes and embeds a single review to understand the difference.

In [ ]:
# Analyze a single review to understand token-level embeddings
sample_review = documents[0]  # Mario's Bistro review
print(f"🔍 Analyzing sample review: {sample_review[:100]}...")

# Create embeddings for the sample review
print("\n⚡ Creating ColBERT embeddings...")
sample_embeddings = colbert_model.encode([sample_review], is_query=False)

# Extract embeddings for analysis
sample_tensor = sample_embeddings[0]  # First (and only) document
num_tokens, embedding_dim = sample_tensor.shape

print(f"📏 Review length: {len(sample_review.split())} words")
print(f"🎯 Number of tokens: {num_tokens}")
print(f"📐 Embedding dimension: {embedding_dim}")
print(f"💾 Total embeddings: {num_tokens * embedding_dim} floats")

# Compare with dense approach
dense_size = 384  # from notebook 2
colbert_size = num_tokens * embedding_dim
size_ratio = colbert_size / dense_size

print(f"\n📊 Size Comparison:")
print(f"   Dense embedding: {dense_size} floats")
print(f"   ColBERT embeddings: {colbert_size} floats")
print(f"   ColBERT is {size_ratio:.1f}x larger but {size_ratio:.1f}x more detailed!")

## 5. Embed All Restaurant Reviews

Now let's create ColBERT embeddings for all restaurant reviews. Each review will have multiple token-level embeddings.

In [ ]:
# Create ColBERT embeddings for all reviews
print(f"🚀 Creating ColBERT embeddings for {len(documents)} reviews...")
start_time = time.time()

# Encode all documents
all_embeddings = colbert_model.encode(documents, is_query=False)

embedding_time = time.time() - start_time
print(f"✅ Embedding completed in {embedding_time:.2f} seconds")
print(f"⚡ Average: {embedding_time/len(documents):.3f} seconds per review")

# Analyze the embedding structure
total_tokens = sum(emb.shape[0] for emb in all_embeddings)
avg_tokens = total_tokens / len(all_embeddings)
min_tokens = min(emb.shape[0] for emb in all_embeddings)
max_tokens = max(emb.shape[0] for emb in all_embeddings)

print(f"\n📊 Embedding Statistics:")
print(f"   Total tokens across all reviews: {total_tokens:,}")
print(f"   Average tokens per review: {avg_tokens:.1f}")
print(f"   Min tokens in a review: {min_tokens}")
print(f"   Max tokens in a review: {max_tokens}")
print(f"   Embedding dimension: {all_embeddings[0].shape[1]}")

# Memory usage estimation
total_floats = sum(emb.shape[0] * emb.shape[1] for emb in all_embeddings)
memory_mb = (total_floats * 4) / (1024 * 1024)  # 4 bytes per float32
print(f"   Estimated memory: {memory_mb:.1f} MB")

## 6. Store ColBERT Embeddings in LanceDB

LanceDB supports multi-vector storage natively, making it perfect for ColBERT. We'll store each document's token embeddings as a matrix.

In [ ]:
# Setup LanceDB for ColBERT multi-vector storage
vector_store_path = os.getenv('VECTOR_STORE_DIR')
print(f"💾 Setting up LanceDB at: {vector_store_path}")

# Connect to LanceDB
db = lancedb.connect(vector_store_path)
table_name = "colbert_restaurant_reviews"

# Prepare data for LanceDB
# Each row will contain: id, restaurant, review, rating, reviewer, embeddings (multi-vector)
colbert_data = []

for idx, (row, embeddings) in enumerate(zip(df.itertuples(), all_embeddings)):
    # Convert embeddings tensor to list of lists (LanceDB format)
    embeddings_list = embeddings.cpu().numpy().tolist()
    
    record = {
        "id": int(row.id),
        "restaurant": row.restaurant,
        "review": row.review,
        "reviewer": row.reviewer,
        "rating": int(row.rating),
        "embeddings": embeddings_list,  # Multi-vector: list of 384-dim vectors
        "num_tokens": len(embeddings_list)
    }
    colbert_data.append(record)

print(f"📦 Prepared {len(colbert_data)} records for storage")
print(f"📏 Sample record structure:")
sample_record = colbert_data[0]
print(f"   ID: {sample_record['id']}")
print(f"   Restaurant: {sample_record['restaurant']}")
print(f"   Review length: {len(sample_record['review'])} chars")
print(f"   Embeddings: {sample_record['num_tokens']} vectors of {len(sample_record['embeddings'][0])} dimensions")

In [ ]:
# Create or recreate the ColBERT table
if table_name in db.table_names():
    print(f"🗑️  Dropping existing table: {table_name}")
    db.drop_table(table_name)

print(f"🏗️  Creating ColBERT table: {table_name}")
colbert_table = db.create_table(table_name, colbert_data)

print(f"✅ ColBERT table created successfully!")
print(f"📊 Table info: {len(colbert_table)} records")

# Verify the storage
sample_result = colbert_table.head(1).to_pandas().iloc[0]
stored_embeddings = np.array(sample_result['embeddings'])
print(f"🔍 Stored embeddings shape: {stored_embeddings.shape}")
print(f"✅ Multi-vector storage verified!")

## 7. Implement ColBERT Search with MaxSim

Now for the exciting part - implementing ColBERT's **MaxSim** operation! This is what makes ColBERT so powerful.

### MaxSim Explained
For each query token:
1. Find its similarity with ALL document tokens
2. Take the **maximum** similarity (best match)
3. Sum these max similarities across all query tokens

This allows fine-grained matching that dense embeddings can't achieve.

In [ ]:
def colbert_search(query: str, table, model, top_k: int = 3) -> List[Dict]:
    """
    Perform ColBERT search using MaxSim operation.
    
    Args:
        query: Search query string
        table: LanceDB table with ColBERT embeddings
        model: ColBERT model for encoding query
        top_k: Number of top results to return
    
    Returns:
        List of search results with scores
    """
    print(f"🔍 ColBERT search: '{query}'")
    
    # Encode query with ColBERT
    query_embeddings = model.encode([query], is_query=True)[0]  # Shape: [query_tokens, dim]
    print(f"📝 Query tokens: {query_embeddings.shape[0]}")
    
    # Get all documents from table
    all_docs = table.to_pandas()
    
    results = []
    
    # Compute MaxSim for each document
    for idx, row in all_docs.iterrows():
        doc_embeddings = torch.tensor(row['embeddings'], dtype=torch.float32)  # Shape: [doc_tokens, dim]
        
        # Compute similarity matrix: [query_tokens, doc_tokens]
        similarity_matrix = torch.matmul(query_embeddings, doc_embeddings.T)
        
        # MaxSim: For each query token, find max similarity with any doc token
        max_similarities = torch.max(similarity_matrix, dim=1)[0]  # Shape: [query_tokens]
        
        # Final score: Sum of max similarities
        score = torch.sum(max_similarities).item()
        
        results.append({
            'id': row['id'],
            'restaurant': row['restaurant'],
            'review': row['review'],
            'reviewer': row['reviewer'],
            'rating': row['rating'],
            'score': score,
            'num_tokens': row['num_tokens']
        })
    
    # Sort by score (descending)
    results.sort(key=lambda x: x['score'], reverse=True)
    
    print(f"✅ Found {len(results)} results, returning top {top_k}")
    return results[:top_k]

def display_colbert_results(results: List[Dict], query: str):
    """
    Display ColBERT search results in a readable format.
    """
    print(f"\n🎯 ColBERT Results for: '{query}'")
    print("=" * 80)
    
    for i, result in enumerate(results, 1):
        print(f"\n{i}. {result['restaurant']} (⭐ {result['rating']}/5)")
        print(f"   Score: {result['score']:.3f} | Tokens: {result['num_tokens']} | Reviewer: {result['reviewer']}")
        print(f"   Review: {result['review'][:200]}{'...' if len(result['review']) > 200 else ''}")
        print("-" * 40)

print("✅ ColBERT search functions defined!")

## 8. Test ColBERT Search

Let's test our ColBERT implementation with the same queries we used in notebook 2 to see how it compares to dense retrieval.

In [ ]:
# Test 1: Simple keyword search
query1 = "Italian pasta authentic"
results1 = colbert_search(query1, colbert_table, colbert_model, top_k=3)
display_colbert_results(results1, query1)

In [ ]:
# Test 2: Work environment query
query2 = "good place to work laptop coding wifi"
results2 = colbert_search(query2, colbert_table, colbert_model, top_k=3)
display_colbert_results(results2, query2)

In [ ]:
# Test 3: Complex contextual query that should showcase ColBERT's power
query3 = "expensive fine dining special occasion worth the money"
results3 = colbert_search(query3, colbert_table, colbert_model, top_k=3)
display_colbert_results(results3, query3)

In [ ]:
# Test 4: Family-friendly search
query4 = "family kids children friendly large groups"
results4 = colbert_search(query4, colbert_table, colbert_model, top_k=3)
display_colbert_results(results4, query4)

## 9. ColBERT Embedding Analysis

Let's analyze the token-level embeddings to understand what ColBERT captures that dense embeddings miss.

In [ ]:
# Analyze token distributions
token_counts = [record['num_tokens'] for record in colbert_data]

plt.figure(figsize=(12, 5))

# Token count distribution
plt.subplot(1, 2, 1)
plt.hist(token_counts, bins=10, alpha=0.7, color='skyblue')
plt.xlabel('Number of Tokens')
plt.ylabel('Number of Reviews')
plt.title('Distribution of Token Counts\nper Review')
plt.grid(True, alpha=0.3)

# Add statistics
plt.axvline(np.mean(token_counts), color='red', linestyle='--', label=f'Mean: {np.mean(token_counts):.1f}')
plt.axvline(np.median(token_counts), color='orange', linestyle='--', label=f'Median: {np.median(token_counts):.1f}')
plt.legend()

# Tokens vs Review Length
review_lengths = [len(record['review']) for record in colbert_data]
plt.subplot(1, 2, 2)
plt.scatter(review_lengths, token_counts, alpha=0.7, color='coral')
plt.xlabel('Review Length (characters)')
plt.ylabel('Number of Tokens')
plt.title('Tokens vs Review Length')
plt.grid(True, alpha=0.3)

# Add correlation
correlation = np.corrcoef(review_lengths, token_counts)[0, 1]
plt.text(0.05, 0.95, f'Correlation: {correlation:.3f}', transform=plt.gca().transAxes, 
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.show()

print(f"📊 Token Statistics:")
print(f"   Average tokens per review: {np.mean(token_counts):.1f}")
print(f"   Standard deviation: {np.std(token_counts):.1f}")
print(f"   Min tokens: {min(token_counts)}")
print(f"   Max tokens: {max(token_counts)}")
print(f"   Correlation with review length: {correlation:.3f}")

## 10. Storage Comparison: Dense vs ColBERT

Let's compare the storage requirements and information density between dense embeddings and ColBERT.

In [ ]:
# Calculate storage requirements
num_reviews = len(colbert_data)
embedding_dim = 384

# Dense embeddings: 1 vector per review
dense_vectors = num_reviews * embedding_dim
dense_memory_mb = (dense_vectors * 4) / (1024 * 1024)  # 4 bytes per float32

# ColBERT embeddings: multiple vectors per review
colbert_vectors = sum(record['num_tokens'] * embedding_dim for record in colbert_data)
colbert_memory_mb = (colbert_vectors * 4) / (1024 * 1024)

# Efficiency metrics
storage_ratio = colbert_memory_mb / dense_memory_mb
info_density_ratio = storage_ratio  # More storage = more information preserved

print(f"💾 Storage Comparison:")
print(f"   Dense embeddings:   {dense_memory_mb:.2f} MB ({dense_vectors:,} vectors)")
print(f"   ColBERT embeddings: {colbert_memory_mb:.2f} MB ({colbert_vectors:,} vectors)")
print(f"   ColBERT uses {storage_ratio:.1f}x more storage")
print(f"   But preserves {storage_ratio:.1f}x more token-level information!")

print(f"\n🎯 Information Density:")
avg_tokens = np.mean(token_counts)
print(f"   Average tokens per review: {avg_tokens:.1f}")
print(f"   Each token gets its own {embedding_dim}-dim vector")
print(f"   vs Dense: entire review → single {embedding_dim}-dim vector")
print(f"   Information preservation: {avg_tokens:.1f}x better!")

# Visualization
plt.figure(figsize=(10, 6))

# Storage comparison bar chart
plt.subplot(1, 2, 1)
methods = ['Dense\nEmbedding', 'ColBERT\nEmbedding']
storage_sizes = [dense_memory_mb, colbert_memory_mb]
colors = ['lightblue', 'lightcoral']

bars = plt.bar(methods, storage_sizes, color=colors, alpha=0.7)
plt.ylabel('Storage Size (MB)')
plt.title('Storage Requirements\nComparison')
plt.grid(True, alpha=0.3)

# Add value labels on bars
for bar, size in zip(bars, storage_sizes):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'{size:.2f} MB', ha='center', va='bottom')

# Information preservation comparison
plt.subplot(1, 2, 2)
info_levels = [1, avg_tokens]  # Dense = 1 vector, ColBERT = avg_tokens vectors
bars = plt.bar(methods, info_levels, color=colors, alpha=0.7)
plt.ylabel('Information Vectors per Review')
plt.title('Information Preservation\nComparison')
plt.grid(True, alpha=0.3)

# Add value labels
for bar, level in zip(bars, info_levels):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
             f'{level:.1f}x', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 11. Save ColBERT Configuration

Save the ColBERT setup for use in the next notebook where we'll compare search quality.

In [ ]:
# Save ColBERT configuration and metadata
colbert_config = {
    "model_name": model_name,
    "embedding_dimension": embedding_dim,
    "device": device,
    "table_name": table_name,
    "num_reviews": num_reviews,
    "total_tokens": sum(token_counts),
    "avg_tokens_per_review": float(np.mean(token_counts)),
    "storage_mb": float(colbert_memory_mb),
    "created_at": pd.Timestamp.now().isoformat()
}

# Save to JSON for next notebook
config_path = os.path.join(os.getenv('PROJECT_ROOT'), 'colbert_config.json')
with open(config_path, 'w') as f:
    json.dump(colbert_config, f, indent=2)

print(f"💾 ColBERT configuration saved to: {config_path}")
print(f"✅ Ready for notebook 4: Search Comparison!")

# Display final summary
print(f"\n🎉 ColBERT Setup Complete!")
print(f"   📊 Embedded {num_reviews} restaurant reviews")
print(f"   🎯 {sum(token_counts):,} total token embeddings")
print(f"   💾 {colbert_memory_mb:.1f} MB storage used")
print(f"   ⚡ {avg_tokens:.1f}x more detailed than dense embeddings")
print(f"   🔍 Ready for MaxSim-powered search!")

## Summary: Why ColBERT is Revolutionary 🚀

In this notebook, we implemented ColBERT's **late interaction** approach and saw how it fundamentally changes retrieval:

### Key Insights:
1. **Token-Level Granularity**: Every word gets its own embedding, preserving nuanced meaning
2. **MaxSim Operation**: Smart token-to-token matching finds the best connections
3. **Information Preservation**: ~40x more detailed than dense embeddings
4. **Storage Trade-off**: Uses more space but captures more context

### What's Next:
In **Notebook 4**, we'll do head-to-head comparisons between dense retrieval and ColBERT to see where each approach excels.

### The Magic of Late Interaction:
Instead of compressing "romantic Italian pasta authentic affordable" into one vector, ColBERT keeps each concept separate and lets them find their best matches in the document. This is why it excels at complex, multi-faceted queries!

---
*Ready to see ColBERT crush traditional RAG? Let's move to the comparison notebook! 🥊*